# 2. Tool Usage & Agentic Workflows

A practical guide to defining and using tools in LangChain:
- **`@tool` Decorator**: Defining custom Python tools with type hints and docstrings
- **`create_agent`**: Creating autonomous agents with LangGraph-backed state orchestration
- **Agent Invocation**: Inspecting message traces through intermediate tool executions
- **Manual Tool Binding**: Binding tools directly to models with `bind_tools`
- **Manual Execution Loop**: Parsing `ai_msg.tool_calls` and driving the tool execution loop explicitly

In [ ]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

@tool
def get_weather(city: str) -> str:
    """Get the current weather forecast for a given city."""
    return f"The weather in {city} is sunny and 24°C."

# Initialize model
llm = ChatOllama(model="qwen3:8b", temperature=0, reasoning=False)
print(f"Tool '{get_weather.name}' created: {get_weather.description}")

### 1. Defining Custom Tools with `@tool`

- **Decorator Pattern**: Uses `@tool` from `langchain_core.tools` to convert standard Python functions into callable schemas.
- **Schema Metadata**: Function name, docstrings, and type annotations are converted into JSON schemas for LLM tool selection and parameter validation.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use tools when needed."
)

agent

### 2. Autonomous Agent Creation (`create_agent`)

- **Agent Compilation**: Uses `create_agent` to assemble the LLM, toolset, and system prompt into a compiled state graph.
- **Automatic Execution**: The agent automatically orchestrates deciding when to call tools, executing them, and synthesizing final answers.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is the weather in Pune?"}]})

for msg in result["messages"]:
    print(f"[{msg.__class__.__name__}]: {msg.content if hasattr(msg, 'content') else msg}")

### 3. Agent Invocation & Message Tracing

- **Execution Flow**: Evaluates the user query, generates an `AIMessage` with tool calls, executes `get_weather`, and receives a `ToolMessage`.
- **Final Output**: The agent incorporates the tool output to construct the final conversational answer.

In [ ]:
messages = [{"role": "user", "content": "What is the weather in Pune?"}]

model_with_tools = llm.bind_tools([get_weather])
ai_msg = model_with_tools.invoke(messages)

print("Tool calls generated by model:")
print(ai_msg.tool_calls)

### 4. Manual Tool Binding (`bind_tools`)

- **Schema Attachment**: Binds tool definitions directly to the model instance via `llm.bind_tools([get_weather])`.
- **Structured Tool Call**: The model inspects the user message and emits structured tool invocations in `ai_msg.tool_calls`.

In [ ]:
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_message = get_weather.invoke(tool_call)
    messages.append(tool_message)

final_resp = model_with_tools.invoke(messages)
print("Final Synthesized Answer:")
print(final_resp.text)

### 5. Manual Tool Execution Loop

- **Explicit Orchestration**: Iterates over `ai_msg.tool_calls`, invokes the Python tool function, and appends the resulting `ToolMessage` to the message history.
- **Grounded Answer**: Re-invokes the model with complete tool results to generate the final natural language answer.